# TV-Denoising Network: Hybrid Deep-Learning & Convex Optimization

A PyTorch implementation of **Differentiable Total Variation Denoising** using `cvxpylayers`. 

`cvxpylayers` is a package developed by Agrawal et al. and introduced in ["Differentiable Convex Optimization Layers"](https://arxiv.org/abs/1910.12449). It allows convex problems that are **DPP-compliant**, meaning they are constructed following the **Disciplined Parametrized Programming** grammar, to be integrated into a Neural Network as a native layer. This enables end-to-end backpropagation through the optimization solver using implicit differentiation.

In this notebook, we load pre-trained deep-learning models containing optimization layers to study different approaches to Total Variation (TV) Denoising on the DIV2K dataset. The dataset is converted to grayscale, and we manually add Additive White Gaussian Noise using the `src.data.dataset.GaussianDenoisingDataset` utility.

### Model Architecture
The model is a hybrid system composed of:
* **CNN Backbone (Weight Predictor):** Analyzes local image features and noise levels to predict a spatially-adaptive $\Lambda$ (lambda) map.
* **CVXPY Optimization Layer:** Solves the TV proximal operator using the predicted $\Lambda$ map as the regularization weight for each pixel.

The backbone outputs a regularization map corresponding to either **"isotropic"** or **"anisotropic"** Total Variation penalties.

### Training Objectives
Models are trained using a **dual-loss** function: 
$$\mathcal{L} = \alpha \cdot \text{MSE} + (1 - \alpha) \cdot (1 - \text{SSIM})$$

This ensures the model does not merely learn to match the ground truth pixel-by-pixel (MSE) but also maintains structural integrity and perceptual quality by mimicking the human visual system (HVS).

---

### Experimental Configurations
Training was conducted across several configurations to evaluate solver efficiency and denoising quality:

1.  **Anisotropic Regularization** with the **SCS** (First-order Splitting Conic) solver.
2.  **Anisotropic Regularization** with the **CLARABEL** (Interior Point Method) solver.
3.  **Isotropic Regularization** with the **SCS** solver.
4.  **Isotropic Regularization** with the **CLARABEL** solver.

#### Numerical Observations
Due to the mathematical coupling of horizontal and vertical gradients in the isotropic formulation (solved as a Second-Order Cone Program), **isotropic training is effectively 5 to 10 times slower** than anisotropic training (solved as a Quadratic Program). Consequently, isotropic models were trained for fewer iterations. 

Furthermore, during isotropic training, we observed that the model struggled to converge initially. This necessitated specific architectural adjustments, including weight initialization of the CNN backbone and architectural changes to handle boundary artifacts, to facilitate stable convergence. However, althought results are slightly better, the loss seemed to stagnate after 3 epochs for isotropic regularization based modelswhereas for reference the anisotropic training on SCS ran for around 50 epochs.


#### Training not part of notebook

Due to the long time needed for training (1 model could take up to dozens of hours), I have decided not to include training part in the notebook. If the reader wishes to train the model himself / change configurations and try out other approaches, please refer to: (https://github.com/Kh-T5/Diff-TV-Net) & the corresponding README.

## Imports

In [ ]:
import os
import sys
notebook_dir = os.path.abspath('')
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
### project related imports
from src.models.nn_hybrid import TVDenoisingNet
from src.data.dataset import GaussianDenoisingDataset
from src.utils.evaluation import evaluate, save_debug_plot
from src.config import (
    data_dir,
    gaussian_std,
    patch_size,
    device_name,
    results_path,
    model_path,
    sample_path,
    sample_dir,
    plots_dir,
    scs_info,
    CLARABEL_info
)

### Visualization imports 
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import time

### Pytorch imports
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

### Convex Optimization imports
import cvxpy as cp

### Metrics
from piq import ssim


## Model loading

In [ ]:
### Samples
samples_dir = r'../data/sample/benchmarking_samples/'
### Models
clarabel_iso = r"../results/models/isotropic/model_clarabel_solver_isotropic.pth"
clarabel_ani = r"../results/models/anisotropic/model_clarabel_solver_anisotropic.pth"
scs_iso = r"../results/models/isotropic/model_scs_solver_low_init_isotropic.pth"
scs_ani = r"../results/models/anisotropic/model_scs_solver_anisotropic.pth"

In [ ]:
def load_benchmarking_models(img_size=(64, 64), device="cpu"):
    """
    Loads all 4 variants of the TV-Denoising Network into a dictionary.
    """
    configs = {
        "anisotropic_scs": {"reg": "anisotropic", "solver": "SCS", "file": scs_ani},
        "anisotropic_clarabel": {"reg": "anisotropic", "solver": "CLARABEL", "file": clarabel_ani},
        "isotropic_scs": {"reg": "isotropic", "solver": "SCS", "file": scs_iso},
        "isotropic_clarabel": {"reg": "isotropic", "solver": "CLARABEL", "file": clarabel_iso},
    }
    
    # Solver settings used during training
    solver_params = {
        "SCS": scs_info,
        "CLARABEL": CLARABEL_info
    }

    loaded_models = {}

    for name, cfg in configs.items():
        path = cfg["file"]
       
            
        model = TVDenoisingNet(
            reg=cfg["reg"], 
            solver=cfg["solver"], 
            solver_info=solver_params[cfg["solver"]],
            img_size=img_size
        )
        
        # Load saved model weights
        state_dict = torch.load(path, map_location=device, weights_only=True)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval() 
        
        loaded_models[name] = model
        print(f"Loaded {name} from {cfg['file']}")
        
    return loaded_models

In [ ]:
loaded_models = load_benchmarking_models()

## Benchmark utils

#### Benchmark functions for baseline solvers and our trained models

Since `cvxpylayers` performs canonicalization once during initialization, the forward pass is theoretically much faster than standard CVXPY for repeated solves.

In [ ]:
class TVBenchmarker:
    """
    Benchmarking class, used to compare baseline solvers from cvxpy 
    with the saved trained models using cvxpylayers and pytorch
    """
    def __init__(self, reg_type: str, solver_name: str):
        model_name = f"{reg_type}_{solver_name}"
        self.model = loaded_models[model_name]
        self.reg = reg_type
        self.solver_name = solver_name
        self.solver_map = {"SCS": cp.SCS, "CLARABEL": cp.CLARABEL}
        
    def run_pure_cvxpy(self, noisy_np, lam_scalar):
        """Standard CVXPY: modeling + canonicalization + solve."""
        h, w = noisy_np.shape
        U = cp.Variable((h, w))
        
        if self.reg == "anisotropic":
            ux = U[1:, :] - U[:-1, :]
            uy = U[:, 1:] - U[:, :-1]
            reg_term = cp.sum(cp.abs(ux)) + cp.sum(cp.abs(uy))
        elif self.reg == "isotropic":
            reg_term = cp.tv(U)
        else:
            raise KeyError(f"Reg not recognized : {self.reg}")
            
        prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(U - noisy_np) + lam_scalar * reg_term))
        
        start = time.perf_counter()
        prob.solve(solver=self.solver_map[self.solver_name])
        elapsed = time.perf_counter() - start
        
        return U.value, elapsed

    def run_hybrid(self, noisy_tensor):
        """Hybrid: CNN inference + pre-compiled cvxpylayer."""
        start = time.perf_counter()
        with torch.no_grad():
            denoised, lam_map = self.model(noisy_tensor)
        elapsed = time.perf_counter() - start
        
        return denoised.squeeze().cpu().numpy(), lam_map.squeeze().cpu().numpy(), elapsed

In [ ]:
def compare_reg_solver(reg, solver, loader, lam):
    """
    Runs inference on DL and baseline models for a given solver for a given regularization method.
    Returns:
    - pd.DataFrame containing run time and SSIM metric output.
    """
    benchmarker = TVBenchmarker(reg, solver)
    results = []
    for id, (noisy, clean) in enumerate(loader):
        
        denoised_h, lam_map, time_h = benchmarker.run_hybrid(noisy)
        
        noisy_np = noisy.squeeze().cpu().numpy()
        denoised_p, time_p = benchmarker.run_pure_cvxpy(noisy_np, lam)
        
        results.append({
            "Image": id,
            "Method": f"Trained model {reg}_{solver}",
            "SSIM": ssim(denoised_h, clean, data_range=1.0),
            "Time (s)": time_h
        })
        results.append({
            "Image": id,
            "Method": f"baseline {solver} with {reg} TV",
            "SSIM": ssim(denoised_p, clean, data_range=1.0),
            "Time (s)": time_p
        })
    return pd.DataFrame(results)

In [ ]:
def visualize_benchmark_results(
        dataloader: DataLoader, 
        reg_type: str, 
        solver_name: str, 
        global_lam: float, 
        num_samples: int=5
        ):
    """
    Visualizes the comparison between Hybrid and Baseline.
    
    Inputs:
        model: The TVDenoisingNet instance.
        dataloader: DataLoader returning (noisy, clean) batches.
        reg_type: "anisotropic" or "isotropic".
        solver_name: "SCS" or "CLARABEL".
        global_lam: Scalar lambda for the standard CVXPY baseline.
    """
    device = "cpu"
    model_name = f"{reg_type}_{solver_name.lower()}"
    model = loaded_models[model_name]
    model.eval()
    
    cp_solvers = {"SCS": cp.SCS, "CLARABEL": cp.CLARABEL}
    
    fig, axes = plt.subplots(num_samples, 5, figsize=(20, 4 * num_samples))
    plt.subplots_adjust(wspace=0.15, hspace=0.3)
    
    cols = ["Ground Truth", "Noisy Input", f"Hybrid ({reg_type})", r"Learned $\Lambda$ Map", "Baseline"]

    with torch.no_grad():
        for i, (noisy, clean) in enumerate(dataloader):
            if i >= num_samples:
                break
            
            noisy, clean = noisy.to(device), clean.to(device)
            
            denoised_h, lam_map = model(noisy)
            
            noisy_np = noisy[0, 0].cpu().numpy()
            U = cp.Variable(noisy_np.shape)
            
            if reg_type == "anisotropic":
                ux = U[1:, :] - U[:-1, :]
                uy = U[:, 1:] - U[:, :-1]
                reg_term = cp.sum(cp.abs(ux)) + cp.sum(cp.abs(uy))
            elif reg_type == "isotropic":
                reg_term = cp.tv(U)
            else:
                raise KeyError(f"Regularization {reg_type} not recognized.")
            
            prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(U - noisy_np) + global_lam * reg_term))
            prob.solve(solver=cp_solvers[solver_name])
            denoised_b = U.value

            display_list = [
                clean[0, 0].cpu().numpy(),
                noisy[0, 0].cpu().numpy(),
                denoised_h[0, 0].cpu().numpy(),
                lam_map[0].cpu().numpy(),
                denoised_b
            ]

            for j, img in enumerate(display_list):
                ax = axes[i, j] if num_samples > 1 else axes[j]
                
                cmap = 'magma' if j == 3 else 'gray'
                im = ax.imshow(img, cmap=cmap)
                
                if j == 3:
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                
                if i == 0:
                    ax.set_title(cols[j], fontweight='bold', fontsize=12)
                ax.axis('off')

    plt.show()

## Benchmark

In [ ]:
dataset = GaussianDenoisingDataset(samples_dir, patch_size=patch_size, sigma=gaussian_std)
loader = DataLoader(dataset, batch_size=1, shuffle=False) 
GLOBAL_LAMBDA = 0.1 
